In [1]:
import pandas as pd
import numpy as np



In [2]:
df = pd.read_csv('../data/csv/processed_data.csv')

In [3]:
# Identify non-feature columns
non_feature_cols = ['tire_number', 'origin', 'measurement_id']
print(f"Non-feature columns: {non_feature_cols}")

# Get numeric columns (wavelength data)
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
wavelength_columns = [col for col in numeric_columns if col not in non_feature_cols]

wavelength_values = [float(col) for col in wavelength_columns]

Non-feature columns: ['tire_number', 'origin', 'measurement_id']


In [4]:
import glob
from scipy.interpolate import interp1d
import os

def load_and_interpolate(filename, wavelength_columns):
    df = pd.read_csv(filename)
    print(f"Loading {filename}: {df.columns.tolist()}")
    
    # Create interpolation function
    f = interp1d(df['Wavelength (nm)'], df['Sum'], kind='linear', fill_value="extrapolate")
    
    # Use actual filename (without path and extension) as column name
    base_filename = os.path.splitext(os.path.basename(filename))[0]
    
    return pd.DataFrame({
        "Wavelength": wavelength_columns, 
        base_filename: f(wavelength_columns)  # Use filename as column name
    })

files = glob.glob("../data/csv/nist/*.csv")
dfs = [load_and_interpolate(f, wavelength_columns=wavelength_columns) for f in files]
combined_df = dfs[0]
for df in dfs[1:]:
    combined_df = combined_df.merge(df, on="Wavelength")

combined_df.set_index("Wavelength", inplace=True)
combined_df.info()

Loading ../data/csv/nist\Aluminium_data.csv: ['Wavelength (nm)', 'Sum', 'Al I (4.9e-02)', 'Al II (9.5e-01)', 'Al III (7.0e-04)']
Loading ../data/csv/nist\calcium_data.csv: ['Wavelength (nm)', 'Sum', 'Ca I (1.7e-02)', 'Ca II (8.9e-01)', 'Ca III (9.1e-02)']
Loading ../data/csv/nist\IJzer_data.csv: ['Wavelength (nm)', 'Sum', 'Fe I (4.4e-02)', 'Fe II (9.5e-01)', 'Fe III (2.2e-03)']
Loading ../data/csv/nist\Kalium_data.csv: ['Wavelength (nm)', 'Sum', 'K I (2.0e-02)', 'K II (9.8e-01)', 'K III (6.0e-09)']
Loading ../data/csv/nist\Koolstof_data.csv: ['Wavelength (nm)', 'Sum', 'C I (6.9e-01)', 'C II (3.1e-01)', 'C III (8.0e-08)']
Loading ../data/csv/nist\Magnesium_data.csv: ['Wavelength (nm)', 'Sum', 'Mg I (3.8e-02)', 'Mg II (9.5e-01)', 'Mg III (8.2e-03)']
Loading ../data/csv/nist\Mangaan_data.csv: ['Wavelength (nm)', 'Sum', 'Mn I (3.3e-02)', 'Mn II (9.6e-01)']
Loading ../data/csv/nist\Natrium_data.csv: ['Wavelength (nm)', 'Sum', 'Na I (2.7e-02)', 'Na II (9.7e-01)', 'Na III (9.7e-16)']
Loading 

In [5]:
combined_df.head()
combined_df.info()
combined_df.index.dtype

<class 'pandas.core.frame.DataFrame'>
Index: 8001 entries, 200.0 to 1000.0
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Aluminium_data  8001 non-null   float64
 1   calcium_data    8001 non-null   float64
 2   IJzer_data      8001 non-null   float64
 3   Kalium_data     8001 non-null   float64
 4   Koolstof_data   8001 non-null   float64
 5   Magnesium_data  8001 non-null   float64
 6   Mangaan_data    8001 non-null   float64
 7   Natrium_data    8001 non-null   float64
 8   Silicium_data   8001 non-null   float64
 9   Stikstof_data   8001 non-null   float64
 10  Waterstof_data  8001 non-null   float64
 11  Zink_data       8001 non-null   float64
 12  Zuurstof_data   8001 non-null   float64
 13  Zwavel_data     8001 non-null   float64
dtypes: float64(14)
memory usage: 937.6+ KB


dtype('O')

In [6]:
combined_df.index.astype(float)

Index([ 200.0,  200.1,  200.2,  200.3,  200.4,  200.5,  200.6,  200.7,  200.8,
        200.9,
       ...
        999.1,  999.2,  999.3,  999.4,  999.5,  999.6,  999.7,  999.8,  999.9,
       1000.0],
      dtype='float64', name='Wavelength', length=8001)

In [7]:
def get_top_wavelengths_per_column(df, top_n=10):
    """
    Extract the top N wavelengths (highest values) for each column in the dataframe
    
    Parameters:
    df: DataFrame with wavelengths as index and elements as columns
    top_n: Number of top wavelengths to extract per column
    
    Returns:
    Dictionary with column names as keys and top wavelengths info as values
    """
    
    results = {}
    
    for column in df.columns:
        # Get the column data and sort by values (descending)
        column_data = df[column].sort_values(ascending=False)
        
        # Get top N wavelengths
        top_wavelengths = column_data.head(top_n)
        
        # Store results with wavelength and intensity values
        results[column] = {
            'wavelengths': top_wavelengths.index.tolist(),
            'intensities': top_wavelengths.values.tolist(),
            'wavelength_intensity_pairs': list(zip(top_wavelengths.index, top_wavelengths.values))
        }
        
        print(f"\n Top {top_n} wavelengths for {column}:")
        print(f"{'Rank':<5} {'Wavelength (nm)':<15} {'Intensity':<12}")
        print("-" * 35)
        
        for i, (wavelength, intensity) in enumerate(top_wavelengths.items(), 1):
            # Handle both string and numeric wavelengths
            try:
                wavelength_float = float(wavelength)
                wavelength_str = f"{wavelength_float:<15.1f}"
            except (ValueError, TypeError):
                wavelength_str = f"{str(wavelength):<15}"
            
            print(f"{i:<5} {wavelength_str} {intensity:<12.2f}")
    
    return results

top_wavelengths_nist = get_top_wavelengths_per_column(combined_df, top_n=10)
print(top_wavelengths_nist)


 Top 10 wavelengths for Aluminium_data:
Rank  Wavelength (nm) Intensity   
-----------------------------------
1     309.3           2914000000.00
2     309.2           2781750000.00
3     309.4           2464200000.00
4     309.1           2163000000.00
5     237.3           1866000000.00
6     396.1           1724800000.00
7     396.2           1723000000.00
8     309.5           1700400000.00
9     237.4           1627500000.00
10    237.2           1525250000.00

 Top 10 wavelengths for calcium_data:
Rank  Wavelength (nm) Intensity   
-----------------------------------
1     393.4           130650000000.00
2     393.3           129300000000.00
3     393.5           124850000000.00
4     393.2           120700000000.00
5     393.6           112900000000.00
6     393.1           106725000000.00
7     393.7           96420000000.00
8     393.0           89132500000.00
9     393.8           77887500000.00
10    392.9           70394999999.99

 Top 10 wavelengths for IJzer_data:
Rank 

In [ ]:
def calculate_top_50_percent_wavelengths(df):
    """
    Calculate which wavelengths account for the top 90% of intensity values for each element.

    Parameters:
    df: DataFrame with wavelengths as index and elements as columns
    
    Returns:
    Dictionary with element names and their top 90% contributing wavelengths
    """
    
    results = {}
    
    for column in df.columns:
        # Get the column data and sort by values (descending)
        column_data = df[column].sort_values(ascending=False)
        
        # Calculate total intensity for this element
        total_intensity = column_data.sum()
        target_intensity = total_intensity * 0.50  # 50% of total
        
        # Find wavelengths that contribute to top 50%
        cumulative_intensity = 0
        top_50_percent_wavelengths = []
        top_50_percent_intensities = []
        
        for wavelength, intensity in column_data.items():
            cumulative_intensity += intensity
            top_50_percent_wavelengths.append(wavelength)
            top_50_percent_intensities.append(intensity)
            
            if cumulative_intensity >= target_intensity:
                break
        
        # Calculate statistics
        wavelength_count = len(top_50_percent_wavelengths)
        percentage_of_total_wavelengths = (wavelength_count / len(column_data)) * 100
        actual_intensity_percentage = (cumulative_intensity / total_intensity) * 100
        
        results[column] = {
            'wavelengths': top_50_percent_wavelengths,
            'intensities': top_50_percent_intensities,
            'wavelength_count': wavelength_count,
            'percentage_of_total_wavelengths': percentage_of_total_wavelengths,
            'actual_intensity_percentage': actual_intensity_percentage,
            'wavelength_range': (min(top_50_percent_wavelengths), max(top_50_percent_wavelengths)),
            'top_5_wavelengths': top_50_percent_wavelengths[:5],  # Show top 5 for summary
            'wavelength_intensity_pairs': list(zip(top_50_percent_wavelengths, top_50_percent_intensities))
        }

        print(f"\n🔬 {column.upper()} - Top 90% Intensity Analysis:")
        print(f"{'='*50}")
        print(f"Wavelengths needed for 90% intensity: {wavelength_count}")
        print(f"Percentage of total wavelengths: {percentage_of_total_wavelengths:.2f}%")
        print(f"Actual intensity captured: {actual_intensity_percentage:.2f}%")
        # Convert wavelengths to float for range display
        wl_floats = [float(wl) for wl in top_50_percent_wavelengths]
        print(f"Wavelength range: {min(wl_floats):.1f} - {max(wl_floats):.1f} nm")
        
        print(f"\nTop 10 most important wavelengths:")
        print(f"{'Rank':<5} {'Wavelength (nm)':<15} {'Intensity':<12} {'Cumulative %':<12}")
        print("-" * 50)
        
        cumulative_for_display = 0
        for i, (wl, intensity) in enumerate(zip(top_50_percent_wavelengths[:10], top_50_percent_intensities[:10]), 1):
            cumulative_for_display += intensity
            cumulative_percent = (cumulative_for_display / total_intensity) * 100
            # Convert wavelength to float for formatting
            try:
                wl_float = float(wl)
                wl_formatted = f"{wl_float:<15.1f}"
            except (ValueError, TypeError):
                wl_formatted = f"{str(wl):<15}"
            print(f"{i:<5} {wl_formatted} {intensity:<12.2f} {cumulative_percent:<12.2f}")
    
    return results

# Calculate top 50% wavelengths for each element
top_50_percent_results = calculate_top_50_percent_wavelengths(combined_df)


🔬 ALUMINIUM_DATA - Top 90% Intensity Analysis:
Wavelengths needed for 90% intensity: 11
Percentage of total wavelengths: 0.14%
Actual intensity captured: 25.77%
Wavelength range: 237.2 - 396.2 nm

Top 10 most important wavelengths:
Rank  Wavelength (nm) Intensity    Cumulative %
--------------------------------------------------
1     309.3           2914000000.00 3.41        
2     309.2           2781750000.00 6.67        
3     309.4           2464200000.00 9.55        
4     309.1           2163000000.00 12.08       
5     237.3           1866000000.00 14.27       
6     396.1           1724800000.00 16.29       
7     396.2           1723000000.00 18.30       
8     309.5           1700400000.00 20.29       
9     237.4           1627500000.00 22.20       
10    237.2           1525250000.00 23.98       

🔬 CALCIUM_DATA - Top 90% Intensity Analysis:
Wavelengths needed for 90% intensity: 5
Percentage of total wavelengths: 0.06%
Actual intensity captured: 25.32%
Wavelength range: 3

In [9]:
element_wavelengths = {}
for element, data in top_50_percent_results.items():
    element_wavelengths[element] = data['wavelengths']

element_wavelengths = pd.DataFrame.from_dict(element_wavelengths, orient='index').transpose()
element_wavelengths.to_csv('../data/csv/nist_top_25_percent_wavelengths.csv', index=False)

In [15]:
nist_top_50_percent_wavelengths = pd.read_csv('../data/csv/nist_top_50_percent_wavelengths.csv')

In [16]:
nist_top_50_percent_wavelengths.head()

,Aluminium_data,calcium_data,IJzer_data,Kalium_data,Koolstof_data,Magnesium_data,Mangaan_data,Natrium_data,Silicium_data,Stikstof_data,Waterstof_data,Zink_data,Zuurstof_data,Zwavel_data
0,309.3,393.4,238.8,766.5,247.9,279.6,257.6,589.1,251.6,868.2,656.3,202.6,777.3,921.4
1,309.2,393.3,238.9,766.4,247.8,279.5,257.7,589.2,251.5,868.1,656.2,202.5,777.4,921.3
2,309.4,393.5,239.0,766.6,248.0,279.7,257.5,589.0,251.7,868.3,656.4,202.4,777.2,921.5
3,309.1,393.2,239.1,766.3,247.7,279.4,259.4,589.3,251.4,868.0,656.1,202.7,777.5,921.2
4,237.3,393.6,238.7,766.7,NaN,279.8,257.8,588.9,251.8,868.4,656.5,206.2,777.1,921.6


### ALUMINIUM_DATA - Top 50% Intensity Analysis:

- Wavelengths needed for 50% intensity: 35
- Percentage of total wavelengths: 0.52%
- Actual intensity captured: 50.37%
- Wavelength range: 185.6 - 396.4 nm


Top 10 most important wavelengths:
| Rank | Wavelength (nm) | Intensity | Cumulative %
| -----| ----------------| ----------| --------------
| 1 | 309.3 | 2914000000.00 | 2.84        
| 2 | 309.2 | 2781750000.00 | 5.54        
| 3 | 309.4 | 2464200000.00 | 7.94        
| 4 | 309.1 | 2163000000.00 | 10.04       
| 5 | 186.2 | 2108800000.00 | 12.10       
| 6 | 186.3 | 1892000000.00 | 13.94       
| 7 | 237.3 | 1866000000.00 | 15.75       
| 8 | 396.1 | 1724800000.00 | 17.43       
| 9 | 396.2 | 1723000000.00 | 19.11       
| 10 | 309.5 | 1700400000.00 | 20.76

### CALCIUM_DATA - Top 50% Intensity Analysis:

- Wavelengths needed for 50% intensity: 13
- Percentage of total wavelengths: 0.19%
- Actual intensity captured: 51.12%
- Wavelength range: 392.9 - 396.9 nm

Top 10 most important wavelengths:
| Rank | Wavelength (nm) | Intensity | Cumulative %
| ----|-------------|--------------|----------------
| 1 | 393.4 | 130650000000.00 | 5.37        
| 2 | 393.3 | 129300000000.00 | 10.69       
| 3 | 393.5 | 124850000000.00 | 15.82       
| 4 | 393.2 | 120700000000.00 | 20.78       
| 5 | 393.6 | 112900000000.00 | 25.42       
| 6 | 393.1 | 106725000000.00 | 29.81       
| 7 | 393.7 | 96420000000.00 | 33.77       
| 8 | 393.0 | 89132500000.00 | 37.44       
| 9 | 393.8 | 77887500000.00 | 40.64       
| 10 | 392.9 | 70394999999.99 | 43.53       

### IJZER_DATA - Top 50% Intensity Analysis:
- Wavelengths needed for 50% intensity: 119
- Percentage of total wavelengths: 1.78%
- Actual intensity captured: 50.10%
- Wavelength range: 233.9 - 275.7 nm

Top 10 most important wavelengths:
| Rank | Wavelength (nm) | Intensity | Cumulative %
| -----| ----------------| ----------| --------------
| 1 | 238.8 | 9328000000.00 | 0.59        
| 2 | 238.9 | 9323000000.00 | 1.18        
| 3 | 239.0 | 9318000000.00 | 1.76        
| 4 | 239.1 | 9313000000.00 | 2.35        
| 5 | 238.7 | 9290000000.00 | 2.94        
| 6 | 239.2 | 9287000000.00 | 3.52        
| 7 | 239.3 | 9261000000.00 | 4.11        
| 8 | 238.6 | 9252000000.00 | 4.69        
| 9 | 239.4 | 9225000000.00 | 5.27        
| 10 | 239.5 | 9189000000.00 | 5.85        

### KALIUM_DATA - Top 50% Intensity Analysis:

- Wavelengths needed for 50% intensity: 11
- Percentage of total wavelengths: 0.16%
- Actual intensity captured: 51.39%
- Wavelength range: 765.9 - 769.9 nm

Top 10 most important wavelengths:
| Rank | Wavelength (nm) | Intensity | Cumulative %
| -----| ----------------| ----------| --------------
| 1 | 766.5 | 358150000.00 | 5.99        
| 2 | 766.4 | 353350000.00 | 11.89       
| 3 | 766.3 | 337100000.00 | 17.53       
| 4 | 766.7 | 332320000.00 | 23.08       
| 5 | 766.8 | 304300000.00 | 28.17       
| 6 | 766.1 | 276540000.00 | 32.79       
| 7 | 766.9 | 269200000.00 | 37.29       
| 8 | 766.0 | 238120000.00 | 41.27       
| 9 | 767.0 | 230200000.00 | 45.12       
| 10 | 765.9 | 198200000.00 | 48.43       

### KOOLSTOF_DATA - Top 50% Intensity Analysis:

- Wavelengths needed for 50% intensity: 3
- Percentage of total wavelengths: 0.04%
- Actual intensity captured: 67.28%
- Wavelength range: 193.0 - 193.2 nm

Top 10 most important wavelengths:
| Rank | Wavelength (nm) | Intensity | Cumulative %
| -----| ----------------| ----------| --------------
| 1 | 193.1 | 4546400000.00 | 26.44       
| 2 | 193.0 | 3691800000.00 | 47.90       
| 3 | 193.2 | 3331600000.00 | 67.28       

### MAGNESIUM_DATA - Top 50% Intensity Analysis:

- Wavelengths needed for 50% intensity: 6
- Percentage of total wavelengths: 0.09%
- Actual intensity captured: 55.37%
- Wavelength range: 279.4 - 280.3 nm

Top 10 most important wavelengths:
| Rank | Wavelength (nm) | Intensity | Cumulative %
| -----| ----------------| ----------| --------------
| 1 | 279.6 | 404979999999.97 | 12.09       
| 2 | 279.5 | 402500000000.00 | 24.10       
| 3 | 279.7 | 322920000000.03 | 33.74       
| 4 | 279.4 | 310849999999.95 | 43.01       
| 5 | 279.8 | 212479999999.98 | 49.35       
| 6 | 280.3 | 201600000000.00 | 55.37       

### MANGAAN_DATA - Top 50% Intensity Analysis:

- Wavelengths needed for 50% intensity: 32
- Percentage of total wavelengths: 0.48%
- Actual intensity captured: 50.36%
- Wavelength range: 257.2 - 295.0 nm

Top 10 most important wavelengths:
| Rank | Wavelength (nm) | Intensity | Cumulative %
| -----| ----------------| ----------| --------------
| 1 | 257.6 | 58582857142.86 | 3.07        
| 2 | 257.7 | 55637142857.14 | 5.98        
| 3 | 257.5 | 54026666666.67 | 8.81        
| 4 | 259.4 | 46675000000.00 | 11.26       
| 5 | 257.8 | 46580000000.00 | 13.70       
| 6 | 259.3 | 45707142857.14 | 16.09       
| 7 | 257.4 | 43918571428.57 | 18.39       
| 8 | 259.5 | 41928571428.57 | 20.59       
| 9 | 259.2 | 39764285714.28 | 22.67       
| 10 | 260.6 | 34595714285.71 | 24.48       

### NATRIUM_DATA - Top 50% Intensity Analysis:

- Wavelengths needed for 50% intensity: 7
- Percentage of total wavelengths: 0.10%
- Actual intensity captured: 50.24%
- Wavelength range: 588.8 - 589.6 nm

Top 10 most important wavelengths:
| Rank | Wavelength (nm) | Intensity | Cumulative %
| -----| ----------------| ----------| --------------
| 1 | 589.1 | 1635600000.00 | 8.18        
| 2 | 589.2 | 1625200000.00 | 16.31       
| 3 | 589.3 | 1559000000.00 | 24.11       
| 4 | 588.9 | 1471400000.00 | 31.47       
| 5 | 588.8 | 1306000000.00 | 38.00       
| 6 | 589.5 | 1305000000.00 | 44.52       
| 7 | 589.6 | 1142750000.00 | 50.24       

### SILICIUM_DATA - Top 50% Intensity Analysis:

- Wavelengths needed for 50% intensity: 21
- Percentage of total wavelengths: 0.31%
- Actual intensity captured: 50.45%
- Wavelength range: 184.6 - 288.3 nm

Top 10 most important wavelengths:
| Rank | Wavelength (nm) | Intensity | Cumulative %
| -----| ----------------| ----------| --------------
| 1 | 251.6 | 5484400000.00 | 4.24        
| 2 | 251.5 | 4961000000.00 | 8.08        
| 3 | 251.7 | 4803000000.00 | 11.80       
| 4 | 185.1 | 4040000000.00 | 14.93       
| 5 | 185.0 | 3940250000.00 | 17.97       
| 6 | 251.4 | 3499250000.00 | 20.68       
| 7 | 251.8 | 3451600000.00 | 23.35       
| 8 | 184.9 | 3384000000.00 | 25.97       
| 9 | 184.8 | 3266750000.00 | 28.50       
| 10 | 212.4 | 3077000000.00 | 30.88

### STIKSTOF_DATA - Top 50% Intensity Analysis:

- Wavelengths needed for 50% intensity: 75
- Percentage of total wavelengths: 1.12%
- Actual intensity captured: 50.10%
- Wavelength range: 746.6 - 939.9 nm

Top 10 most important wavelengths:
| Rank | Wavelength (nm) | Intensity | Cumulative %
| -----| ----------------| ----------| --------------
| 1 | 868.2 | 3708800.00 | 1.69        
| 2 | 868.1 | 3672200.00 | 3.36        
| 3 | 868.0 | 3551000.00 | 4.98        
| 4 | 868.4 | 3529000.00 | 6.58        
| 5 | 867.9 | 3355000.00 | 8.11        
| 6 | 868.5 | 3322750.00 | 9.63        
| 7 | 868.6 | 3059500.00 | 11.02       
| 8 | 867.7 | 2782800.00 | 12.29       
| 9 | 868.7 | 2754750.00 | 13.54       
| 10 | 867.6 | 2445800.00 | 14.65       

### WATERSTOF_DATA - Top 50% Intensity Analysis:

- Wavelengths needed for 50% intensity: 8
- Percentage of total wavelengths: 0.12%
- Actual intensity captured: 50.53%
- Wavelength range: 655.8 - 656.8 nm

Top 10 most important wavelengths:
| Rank | Wavelength (nm) | Intensity | Cumulative %
| -----| ----------------| ----------| --------------
| 1 | 656.2 | 27022000.00 | 7.79        
| 2 | 656.4 | 26514000.00 | 15.43       
| 3 | 656.1 | 25460000.00 | 22.77       
| 4 | 656.5 | 24516000.00 | 29.83       
| 5 | 656.0 | 22875000.00 | 36.42       
| 6 | 656.7 | 18222500.00 | 41.68       
| 7 | 655.8 | 16077500.00 | 46.31       
| 8 | 656.8 | 14655000.00 | 50.53       

### ZINK_DATA - Top 50% Intensity Analysis:

- Wavelengths needed for 50% intensity: 5
- Percentage of total wavelengths: 0.07%
- Actual intensity captured: 51.45%
- Wavelength range: 202.4 - 206.2 nm

Top 10 most important wavelengths:
| Rank | Wavelength (nm) | Intensity | Cumulative %
| -----| ----------------| ----------| --------------
| 1 | 202.5 | 200720000000.00 | 13.59       
| 2 | 202.6 | 200720000000.00 | 27.17       
| 3 | 202.4 | 124900000000.01 | 35.63       
| 4 | 202.7 | 124820000000.02 | 44.08       
| 5 | 206.2 | 108880000000.00 | 51.45       

### ZUURSTOF_DATA - Top 50% Intensity Analysis:

- Wavelengths needed for 50% intensity: 19
- Percentage of total wavelengths: 0.28%
- Actual intensity captured: 50.75%
- Wavelength range: 776.5 - 926.5 nm

Top 10 most important wavelengths:
| Rank | Wavelength (nm) | Intensity | Cumulative %
| -----| ----------------| ----------| --------------
| 1 | 777.3 | 10330000.00 | 4.29        
| 2 | 777.4 | 10290000.00 | 8.57        
| 3 | 777.5 | 9960600.00 | 12.71       
| 4 | 777.1 | 9568200.00 | 16.69       
| 5 | 777.0 | 8828600.00 | 20.36       
| 6 | 777.7 | 8591200.00 | 23.93       
| 7 | 776.9 | 7922800.00 | 27.22       
| 8 | 777.8 | 7650400.00 | 30.40       
| 9 | 776.8 | 6913000.00 | 33.27       
| 10 | 777.9 | 6625200.00 | 36.03

### ZWAVEL_DATA - Top 50% Intensity Analysis:

- Wavelengths needed for 50% intensity: 24
- Percentage of total wavelengths: 0.36%
- Actual intensity captured: 50.66%
- Wavelength range: 920.7 - 923.5 nm

Top 10 most important wavelengths:
| Rank | Wavelength (nm) | Intensity | Cumulative %
| -----| ----------------| ----------| --------------
| 1 | 921.3 | 22767500.00 | 2.55        
| 2 | 921.5 | 22617500.00 | 5.08        
| 3 | 921.2 | 22270000.00 | 7.57        
| 4 | 921.6 | 22060000.00 | 10.04       
| 5 | 921.1 | 21332000.00 | 12.42       
| 6 | 921.7 | 21308000.00 | 14.81       
| 7 | 921.8 | 20468000.00 | 17.10       
| 8 | 921.0 | 20030000.00 | 19.34       
| 9 | 922.0 | 18898000.00 | 21.45       
| 10 | 920.9 | 18402000.00 | 23.51

Aluminium 186.2 - 186.3 nm, 237.3 nm, 309.1 - 309.5 nm

Calcium 392.9 - 393.8 nm

IJzer 238.6 - 239.5 nm

Kalium 765.9 - 767.0 nm

Koolstof 192.8 - 193.4 nm, 247.8 - 248.0 nm

Magnesium 279.5 - 280.4 nm

Mangaan 257.4 - 257.8 nm, 259.2 - 259.5 nm, 260.6 nm

Natrium 588.5 - 589.8 nm

Silicium 184.8 - 185.1 nm, 158.0 - 158.1 nm, 212.4 nm, 251.4 - 251.8 nm

Stikstof 867.6 - 868.7 nm

Waterstof 655.7 - 656.9 nm

Zink 202.3 - 202.7 nm, 206.1 - 206.3 nm, 219.8 nm, 219.9 nm

Zuurstof 776.8 nm - 777.9 nm

Zwavel 920.9 - 922.0 nm

Deze golflengtebereiken kunnen worden gebruikt om specifieke elementen in de spectrale gegevens te identificeren en te analyseren.

In [9]:
Elemental_Wavelength_Ranges_NIST = {    
    "Aluminium": [(186.2, 186.3), (237.3, 237.3), (309.1, 309.5)],
    "Calcium": [(392.9, 393.8)],
    "IJzer": [(238.6, 239.5)],
    "Kalium": [(765.9, 767.0)],
    "Koolstof": [(192.8, 193.4), (247.8, 248.0)],
    "Magnesium": [(279.5, 280.4)],
    "Mangaan": [(257.4, 257.8), (259.2, 259.5), (260.6, 260.6)],
    "Natrium": [(588.5, 589.8)],
    "Silicium": [(184.8, 185.1), (158.0, 158.1), (212.4, 212.4), (251.4, 251.8)],
    "Stikstof": [(867.6, 868.7)],
    "Waterstof": [(655.7, 656.9)],
    "Zink": [(202.3, 202.7), (206.1, 206.3), (219.8, 219.9)],
    "Zuurstof": [(776.8, 777.9)],
    "Zwavel": [(920.9, 922.0)]
}

alle_nm = [golflengte for bereik in Elemental_Wavelength_Ranges_NIST.values() for duo in bereik for golflengte in duo]

In [ ]:
#export Elemental_wavelength_ranges_nist to a csv of the same nsame
import pandas as pd

df = pd.DataFrame(alle_nm, columns=["Wavelength (nm)"])
df.to_csv("Elemental_Wavelength_Ranges_NIST.csv", index=False)

In [12]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

def create_interactive_emission_lines_plot(elemental_ranges, wavelength_columns=None):
    """
    Create an interactive Plotly plot showing emission lines for all elements
    
    Parameters:
    -----------
    elemental_ranges : dict
        Dictionary with element names as keys and list of wavelength ranges as values
    wavelength_columns : list, optional
        List of available wavelengths in your dataset
    """
    
    # Create figure
    fig = go.Figure()
    
    # Color palette for elements
    colors = px.colors.qualitative.Plotly + px.colors.qualitative.Set2 + px.colors.qualitative.Pastel
    
    # Track all wavelengths for x-axis range
    all_wavelengths = []
    
    # Add emission lines for each element
    for i, (element, ranges) in enumerate(elemental_ranges.items()):
        color = colors[i % len(colors)]
        
        # Extract all wavelengths for this element
        element_wavelengths = []
        for wl_range in ranges:
            start, end = wl_range
            element_wavelengths.extend([start, end])
            all_wavelengths.extend([start, end])
        
        # Create vertical lines for each range
        for j, (start, end) in enumerate(ranges):
            # Calculate midpoint and width
            midpoint = (start + end) / 2
            width = end - start
            
            # Add a vertical span for the range
            if width > 0.5:  # If it's a range
                # Add shaded region
                fig.add_vrect(
                    x0=start, x1=end,
                    fillcolor=color,
                    opacity=0.3,
                    layer="below",
                    line_width=0,
                    annotation_text=element if j == 0 else "",
                    annotation_position="top",
                    name=element,
                    showlegend=False
                )
                
                # Add border lines
                fig.add_vline(
                    x=start, 
                    line_width=2, 
                    line_color=color,
                    opacity=0.8,
                    showlegend=False
                )
                fig.add_vline(
                    x=end, 
                    line_width=2, 
                    line_color=color,
                    opacity=0.8,
                    showlegend=False
                )
            else:  # Single wavelength
                fig.add_vline(
                    x=midpoint, 
                    line_width=3, 
                    line_color=color,
                    opacity=0.8,
                    annotation_text=element if j == 0 else "",
                    annotation_position="top",
                    showlegend=False
                )
            
            # Add trace for legend (invisible scatter point)
            if j == 0:
                fig.add_trace(go.Scatter(
                    x=[midpoint],
                    y=[i+1],
                    mode='markers',
                    marker=dict(size=10, color=color),
                    name=element,
                    showlegend=True,
                    hovertemplate=f"<b>{element}</b><br>" +
                                 f"Range: {start:.1f} - {end:.1f} nm<br>" +
                                 "<extra></extra>"
                ))
    
    # Update layout
    fig.update_layout(
        title={
            'text': "🔬 NIST Elemental Emission Lines - Interactive Visualization",
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20, 'color': '#2c3e50'}
        },
        xaxis_title="Wavelength (nm)",
        yaxis_title="Element Index",
        height=800,
        hovermode='closest',
        plot_bgcolor='white',
        legend=dict(
            title="Elements",
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=0.99,
            bgcolor="rgba(255, 255, 255, 0.8)",
            bordercolor="Black",
            borderwidth=1
        ),
        xaxis=dict(
            showgrid=True,
            gridwidth=1,
            gridcolor='LightGray',
            range=[min(all_wavelengths)-10, max(all_wavelengths)+10]
        ),
        yaxis=dict(
            showticklabels=False,
            showgrid=False
        )
    )
    
    return fig

def create_emission_spectrum_detailed(elemental_ranges, combined_df=None):
    """
    Create a more detailed emission spectrum plot with intensity information
    
    Parameters:
    -----------
    elemental_ranges : dict
        Dictionary with element names as keys and wavelength ranges as values
    combined_df : pd.DataFrame, optional
        DataFrame with wavelength intensities for each element
    """
    
    # Create subplots - one for overview, one for detailed view
    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.6, 0.4],
        subplot_titles=("Emission Lines Overview", "Wavelength Ranges Summary"),
        vertical_spacing=0.12
    )
    
    colors = px.colors.qualitative.Plotly + px.colors.qualitative.Set2
    
    # Plot 1: Emission lines as vertical bars
    for i, (element, ranges) in enumerate(elemental_ranges.items()):
        color = colors[i % len(colors)]
        
        for start, end in ranges:
            midpoint = (start + end) / 2
            
            # Add bar in upper subplot
            fig.add_trace(
                go.Bar(
                    x=[midpoint],
                    y=[1],
                    width=max(end - start, 0.5),
                    name=element,
                    marker_color=color,
                    showlegend=True if ranges.index((start, end)) == 0 else False,
                    hovertemplate=f"<b>{element}</b><br>" +
                                 f"Wavelength: {start:.1f} - {end:.1f} nm<br>" +
                                 "<extra></extra>",
                    legendgroup=element
                ),
                row=1, col=1
            )
    
    # Plot 2: Summary table/bars
    element_info = []
    for element, ranges in elemental_ranges.items():
        all_wavelengths = [wl for start, end in ranges for wl in [start, end]]
        element_info.append({
            'Element': element,
            'Min_WL': min(all_wavelengths),
            'Max_WL': max(all_wavelengths),
            'N_Ranges': len(ranges),
            'Total_Coverage': sum([end - start for start, end in ranges])
        })
    
    summary_df = pd.DataFrame(element_info)
    
    # Add horizontal bars for wavelength range coverage
    for i, row in summary_df.iterrows():
        color = colors[i % len(colors)]
        fig.add_trace(
            go.Bar(
                y=[row['Element']],
                x=[row['Total_Coverage']],
                orientation='h',
                name=row['Element'],
                marker_color=color,
                showlegend=False,
                text=f"{row['Total_Coverage']:.1f} nm",
                textposition='auto',
                hovertemplate=f"<b>{row['Element']}</b><br>" +
                             f"Total Coverage: {row['Total_Coverage']:.1f} nm<br>" +
                             f"Range: {row['Min_WL']:.1f} - {row['Max_WL']:.1f} nm<br>" +
                             f"Number of Ranges: {row['N_Ranges']}<br>" +
                             "<extra></extra>",
                legendgroup=row['Element']
            ),
            row=2, col=1
        )
    
    # Update layout
    fig.update_xaxes(title_text="Wavelength (nm)", row=1, col=1)
    fig.update_yaxes(title_text="Intensity (arbitrary)", row=1, col=1, showticklabels=False)
    fig.update_xaxes(title_text="Total Wavelength Coverage (nm)", row=2, col=1)
    fig.update_yaxes(title_text="Element", row=2, col=1)
    
    fig.update_layout(
        title={
            'text': "🔬 NIST Elemental Emission Lines - Detailed Analysis",
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20}
        },
        height=1000,
        showlegend=True,
        legend=dict(
            title="Elements",
            orientation="v",
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=1.15
        ),
        barmode='stack'
    )
    
    return fig

def create_wavelength_heatmap(elemental_ranges):
    """
    Create a heatmap showing which wavelength ranges are covered by each element
    """
    
    # Create a matrix showing presence of elements at different wavelengths
    all_wavelengths = []
    for ranges in elemental_ranges.values():
        for start, end in ranges:
            all_wavelengths.extend([start, end])
    
    wl_min, wl_max = min(all_wavelengths), max(all_wavelengths)
    
    # Create wavelength bins
    n_bins = 200
    wl_bins = np.linspace(wl_min, wl_max, n_bins)
    
    # Create matrix
    matrix = []
    elements = list(elemental_ranges.keys())
    
    for element in elements:
        row = np.zeros(n_bins)
        for start, end in elemental_ranges[element]:
            # Find which bins are covered
            mask = (wl_bins >= start) & (wl_bins <= end)
            row[mask] = 1
        matrix.append(row)
    
    matrix = np.array(matrix)
    
    # Create heatmap
    fig = go.Figure(data=go.Heatmap(
        z=matrix,
        x=wl_bins,
        y=elements,
        colorscale='Viridis',
        showscale=False,
        hovertemplate='Element: %{y}<br>Wavelength: %{x:.1f} nm<br>Present: %{z}<extra></extra>'
    ))
    
    fig.update_layout(
        title={
            'text': "🔬 Elemental Emission Coverage Heatmap",
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20}
        },
        xaxis_title="Wavelength (nm)",
        yaxis_title="Element",
        height=600,
        plot_bgcolor='white'
    )
    
    return fig

# Create the visualizations
print("=" * 70)
print("🎨 CREATING INTERACTIVE EMISSION LINE VISUALIZATIONS")
print("=" * 70)

# Plot 1: Interactive emission lines
fig1 = create_interactive_emission_lines_plot(Elemental_Wavelength_Ranges_NIST, wavelength_values)
fig1.show()

# Plot 2: Detailed spectrum with summary
fig2 = create_emission_spectrum_detailed(Elemental_Wavelength_Ranges_NIST)
fig2.show()

# # Plot 3: Wavelength coverage heatmap
# fig3 = create_wavelength_heatmap(Elemental_Wavelength_Ranges_NIST)
# fig3.show()

# Print summary statistics
print("\n" + "=" * 70)
print("📊 EMISSION LINE STATISTICS")
print("=" * 70)

for element, ranges in Elemental_Wavelength_Ranges_NIST.items():
    all_wl = [wl for start, end in ranges for wl in [start, end]]
    total_coverage = sum([end - start for start, end in ranges])
    
    print(f"\n{element}:")
    print(f"  • Number of ranges: {len(ranges)}")
    print(f"  • Total coverage: {total_coverage:.1f} nm")
    print(f"  • Wavelength span: {min(all_wl):.1f} - {max(all_wl):.1f} nm")
    print(f"  • Ranges: {', '.join([f'{s:.1f}-{e:.1f}' for s, e in ranges])}")

🎨 CREATING INTERACTIVE EMISSION LINE VISUALIZATIONS



📊 EMISSION LINE STATISTICS

Aluminium:
  • Number of ranges: 3
  • Total coverage: 0.5 nm
  • Wavelength span: 186.2 - 309.5 nm
  • Ranges: 186.2-186.3, 237.3-237.3, 309.1-309.5

Calcium:
  • Number of ranges: 1
  • Total coverage: 0.9 nm
  • Wavelength span: 392.9 - 393.8 nm
  • Ranges: 392.9-393.8

IJzer:
  • Number of ranges: 1
  • Total coverage: 0.9 nm
  • Wavelength span: 238.6 - 239.5 nm
  • Ranges: 238.6-239.5

Kalium:
  • Number of ranges: 1
  • Total coverage: 1.1 nm
  • Wavelength span: 765.9 - 767.0 nm
  • Ranges: 765.9-767.0

Koolstof:
  • Number of ranges: 2
  • Total coverage: 0.8 nm
  • Wavelength span: 192.8 - 248.0 nm
  • Ranges: 192.8-193.4, 247.8-248.0

Magnesium:
  • Number of ranges: 1
  • Total coverage: 0.9 nm
  • Wavelength span: 279.5 - 280.4 nm
  • Ranges: 279.5-280.4

Mangaan:
  • Number of ranges: 3
  • Total coverage: 0.7 nm
  • Wavelength span: 257.4 - 260.6 nm
  • Ranges: 257.4-257.8, 259.2-259.5, 260.6-260.6

Natrium:
  • Number of ranges: 1
  • Total 